# `NRChemicalEquilibriumEngine` — Instantiation and Use

Three engines satisfy `ChemicalEquilibriumEngineProtocol`:
`BisectionChemicalEquilibriumEngine`, `NRChemicalEquilibriumEngine`, and
`PHREEQCChemicalEquilibriumEngine`. None is "the" default — pick the one
that matches your chemistry's shape. This notebook covers
`NRChemicalEquilibriumEngine`: a Newton-Raphson solver over a log-linear
tableau in log-activity space. Unlike
`BisectionChemicalEquilibriumEngine` (see
[`01_bisection_engine_basics.ipynb`](01_bisection_engine_basics.ipynb)),
it handles **arbitrary cross-component networks** (species whose mass
balance couples to more than one "total"), **gas-liquid folding**
(Henry/Raoult rows solved simultaneously with the acid-base system,
not diverted), and **solid-liquid precipitation** (Ksp, via a nested
active-set loop).

This notebook focuses on the mechanics of the engine itself:
constructing it, calling `solve()`, and reading back an
`EquilibriumResult` — including the two capabilities
`BisectionChemicalEquilibriumEngine` structurally cannot have. See
[`03_phreeqc_engine_basics.ipynb`](03_phreeqc_engine_basics.ipynb) for
the third engine, which forgoes declared reaction networks entirely.
For a deep accuracy comparison against the Bisection engine and the full
protocol hierarchy (black/gray/white-box), see
[`../../model_api/chemistry/speciation/02_multi_component_systems.ipynb`](../../model_api/chemistry/speciation/02_multi_component_systems.ipynb)
and
[`../../model_api/chemistry/speciation/10_engine_protocol_hierarchy.ipynb`](../../model_api/chemistry/speciation/10_engine_protocol_hierarchy.ipynb).

In [1]:
import sys
from pathlib import Path

def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("Run from inside the PyOMES repo")

sys.path.insert(0, str(_find_repo() / "models"))

from PyOMES.chemistry.common_species import (
    H2O, H_plus, OH_minus,
    CO2, HCO3_minus, CO3_2minus,
    Ca_plus_plus,
)
from PyOMES.chemistry.species import Species
from PyOMES.chemistry import HenryEquilibrium
from PyOMES.reactions.equilibrium import EquilibriumReaction
from PyOMES.reactions.stoichiometry import StoichiometryEntry
from PyOMES.chemical_equilibrium.nr_engine import NRChemicalEquilibriumEngine

def _e(sp, coeff, phase="liquid"):
    return StoichiometryEntry(species=sp, phase=phase, coefficient=coeff)

print("Imports OK")


Imports OK


## 1  Instantiation via `from_reactions()`

`NRChemicalEquilibriumEngine.from_reactions()` takes **one flat list**
of any `EquilibriumConstraint`-conforming item — `EquilibriumReaction`,
`HenryEquilibrium`, `KspEquilibrium`, `RaoultEquilibrium` — and
auto-classifies each via `classify_equilibrium_constraint()`:
acid-base items build the Newton tableau graph; solid-liquid items are
auto-detected as precipitation reactions (no separate
`precipitation_reactions=` kwarg required); gas-liquid items with a
real `log_K` are **folded directly into the tableau** as gas-phase
secondaries. This is the opposite of
`BisectionChemicalEquilibriumEngine`'s behaviour, which diverts
everything except single-phase acid-base items to
`cross_phase_constraints` — see Section 5.

In [2]:
water = EquilibriumReaction(
    stoichiometry=[_e(H2O, -1), _e(H_plus, +1), _e(OH_minus, +1)],
    log_K=-14.0, label="water",
)
co2_first = EquilibriumReaction(
    stoichiometry=[_e(CO2, -1), _e(H2O, -1), _e(HCO3_minus, +1), _e(H_plus, +1)],
    log_K=-6.35, total_id="CO2", label="co2_first",
)
co2_second = EquilibriumReaction(
    stoichiometry=[_e(HCO3_minus, -1), _e(CO3_2minus, +1), _e(H_plus, +1)],
    log_K=-10.33, total_id="CO2", label="co2_second",
)

engine = NRChemicalEquilibriumEngine.from_reactions([water, co2_first, co2_second], T_K=298.15)

print("Engine constructed:  ", type(engine).__name__)
print("Tableau masters:     ", engine._tableau.masters)
print("gas_liquid_species(): ", engine.gas_liquid_species())


Engine constructed:   NRChemicalEquilibriumEngine
Tableau masters:      ['H+', 'CO2']
gas_liquid_species():  frozenset()


## 2  `solve()` and the `totals=` dict

Unlike `BisectionChemicalEquilibriumEngine.solve(CT_TIC=..., CT_NH_T=...)`,
`NRChemicalEquilibriumEngine.solve()` takes totals as a single **`totals=`
dict, keyed by master id** (the species each connected component is
tracked against — see `engine._tableau.masters`), plus a separate
`strong_ions=` dict for non-reactive charge carriers (`CT_Na`, `CT_Cl`, ...).
Both engines return the same `EquilibriumResult` type, but
`charge_residual` — `None` for the Bisection engine — is populated here:
it's literally the last row of the Newton residual, so it always exists
for this solver.

In [3]:
result = engine.solve(totals={"CO2": 0.01}, strong_ions={})   # 10 mmol/L total inorganic carbon

print(f"pH               = {result.pH:.4f}")
print(f"species_mol_L    = {result.species_mol_L}")
print(f"charge_residual  = {result.charge_residual:.2e}  (populated — last Newton residual row)")
print(f"ionic_strength   = {result.ionic_strength:.4e} mol/L")


pH               = 4.1765
species_mol_L    = {'H+': 6.661154515117278e-05, 'CO2': 0.00993338865358094, 'OH-': 1.5012412604009289e-10, 'HCO3-': 6.66113016286192e-05, 'CO3--': 4.6773343131251114e-11, 'H2O': 55.50929780738274}
charge_residual  = -1.48e-13  (populated — last Newton residual row)
ionic_strength   = 1.0000e-02 mol/L


### 2.1  Gotcha: Bisection-style named kwargs are silently ignored

`solve()` only ever reads `totals=`/`strong_ions=`/`phases=`/`T_K=`/
`V_liq_L=`/`V_gas_L=` from its `**kwargs` — a Bisection-style
`CT_TIC=0.01` is accepted syntactically (absorbed into `**kwargs` and
never read) but has no effect; `totals` defaults to `{}` and every
total is `0.0`. Exactly the mirror image of Section 2.1 in the
Bisection notebook.

In [4]:
result_wrong = engine.solve(CT_TIC=0.01)                        # WRONG for this engine
result_right = engine.solve(totals={"CO2": 0.01}, strong_ions={})  # correct

print(f"pH with CT_TIC=0.01 (ignored):        {result_wrong.pH:.4f}  ← defaults to totals={{}}")
print(f"pH with totals={{'CO2': 0.01}} (correct): {result_right.pH:.4f}")


pH with CT_TIC=0.01 (ignored):        7.0000  ← defaults to totals={}
pH with totals={'CO2': 0.01} (correct): 4.1765


## 3  Strong ions and the Davies activity model

Strong ions go in the `strong_ions=` dict (`{"CT_Na": 0.01, ...}`), not
direct kwargs. Activity corrections are configured once at construction
time via `use_activity=`/`activity_model=`, exactly as for
`BisectionChemicalEquilibriumEngine`.

In [5]:
baseline = engine.solve(totals={"CO2": 0.01}, strong_ions={})
dosed    = engine.solve(totals={"CO2": 0.01}, strong_ions={"CT_Na": 0.01})

print(f"pH baseline (no strong ions):  {baseline.pH:.4f}")
print(f"pH with 10 mmol/L Na+ dosing:  {dosed.pH:.4f}")

engine_davies = NRChemicalEquilibriumEngine.from_reactions(
    [water, co2_first, co2_second], use_activity=True, activity_model="davies", T_K=298.15,
)
strong_nacl = {"CT_Na": 0.01, "CT_Cl": 0.01}
ideal_nacl  = engine.solve(totals={"CO2": 0.01}, strong_ions=strong_nacl)
davies_nacl = engine_davies.solve(totals={"CO2": 0.01}, strong_ions=strong_nacl)

print(f"\npH ideal (γ=1),   +10 mmol/L NaCl: {ideal_nacl.pH:.4f}")
print(f"pH Davies-corrected, +10 mmol/L NaCl: {davies_nacl.pH:.4f}")
print(f"Ionic strength (Davies run):          {davies_nacl.ionic_strength*1e3:.2f} mmol/L")


pH baseline (no strong ions):  4.1765
pH with 10 mmol/L Na+ dosing:  8.3353

pH ideal (γ=1),   +10 mmol/L NaCl: 4.1765
pH Davies-corrected, +10 mmol/L NaCl: 4.1766
Ionic strength (Davies run):          10.07 mmol/L


## 4  Solid-liquid: precipitation (Ksp) is auto-classified

A solid-liquid `EquilibriumReaction` — one `StoichiometryEntry(phase="solid")`
(the mineral) plus one or more `phase="liquid"` dissolved products, with
`log_K` set to `log10(Ksp)` — is auto-detected from the same flat list
passed to `from_reactions()` and routed into `engine._precipitation_reactions`,
which drives an outer active-set loop around the inner Newton solve. No
separate `precipitation_reactions=` kwarg is needed (that kwarg still
exists but is deprecated — see `EQUILIBRIUM_CONSTRAINT_UNIFICATION` CP2).

In [6]:
CaCO3_solid = Species(id="CaCO3(s)", atoms={"Ca": 1, "C": 1, "O": 3}, charge=0, MW=100.086)
calcite = EquilibriumReaction(
    stoichiometry=[
        _e(CaCO3_solid, -1, phase="solid"),
        _e(Ca_plus_plus, +1),
        _e(CO3_2minus, +1),
    ],
    log_K=-8.48, label="calcite",   # log10(Ksp) at 25 C
)

engine_precip = NRChemicalEquilibriumEngine.from_reactions(
    [water, co2_first, co2_second, calcite], use_activity=True, activity_model="davies", T_K=298.15,
)
print("Auto-detected precipitation reactions:",
      [r.label for r in engine_precip._precipitation_reactions])

# Supersaturated w.r.t. calcite: high CT_CO2 + Ca2+ dosing
out_precip = engine_precip.solve(
    totals={"CO2": 0.010}, strong_ions={"CT_Ca": 0.002, "CT_Na": 0.005},
)
xi = out_precip.extra["minerals_xi_mol_L"]["calcite"]
si = out_precip.saturation_indices["calcite"]
print(f"xi (mol/L precipitated) = {xi:.6f}")
print(f"saturation index (~0 at equilibrium) = {si:.4f}")


Auto-detected precipitation reactions: ['calcite']


xi (mol/L precipitated) = 0.000552
saturation index (~0 at equilibrium) = -0.0000


## 5  Gas-liquid folding: the capability Bisection cannot have

A fully-parameterized `HenryEquilibrium` (`gas_species`/`liquid_species`
set, so it carries a real `log_K`) is folded directly into the Newton
tableau as a gas-phase secondary attached to the acid-base component its
liquid form belongs to — solved **simultaneously** with the acid-base
system, not as a sequential post-speciation correction
(`KineticGasLiquidLink`'s SNIA path). This is the opposite of
`BisectionChemicalEquilibriumEngine.from_reactions()`, which diverts the
same `HenryEquilibrium` to `cross_phase_constraints` and never solves it
(Section 5 of the Bisection notebook).

Folding a gas-liquid component changes `solve()`'s calling contract:
`totals` for that component must be the **total across both phases**
(mol/L liquid-volume-normalized), and `V_liq_L`/`V_gas_L` (or a
`phases={"liquid":..., "gas":...}` dict) become required. The result
gains a `partial_pressures_atm` field (atm, not mol/L — kept separate
from `species_mol_L` to avoid mislabeling a pressure as a
concentration).

In [7]:
co2_henry = HenryEquilibrium(H_ref=3.4e-4, dlnH=2400.0, gas_species="CO2", liquid_species="CO2",
                              label="co2_henry")

engine_folded = NRChemicalEquilibriumEngine.from_reactions(
    [water, co2_first, co2_second, co2_henry], T_K=298.15,
)
print("gas_liquid_species(): ", engine_folded.gas_liquid_species())

V_liq, V_gas, T_K = 1.0, 0.2, 298.15
n_total_C = 0.05   # mol, across BOTH phases
out_folded = engine_folded.solve(
    totals={"CO2": n_total_C / V_liq}, strong_ions={},
    V_liq_L=V_liq, V_gas_L=V_gas,
)

C_liq_total = (
    out_folded.species_mol_L["CO2"] + out_folded.species_mol_L["HCO3-"]
    + out_folded.species_mol_L["CO3--"]
)
n_liq = C_liq_total * V_liq
n_gas = out_folded.partial_pressures_atm["CO2"] * V_gas / (0.0820574 * T_K)

print(f"pH                        = {out_folded.pH:.4f}")
print(f"partial_pressures_atm     = {out_folded.partial_pressures_atm}")
print(f"n_liq (dissolved C, mol)  = {n_liq:.6f}")
print(f"n_gas (gas-phase C, mol)  = {n_gas:.6f}")
print(f"n_liq + n_gas             = {n_liq + n_gas:.6f}  (must equal n_total_C = {n_total_C})")


gas_liquid_species():  frozenset({'CO2'})
pH                        = 3.8723
partial_pressures_atm     = {'CO2': 1.1698643869148526}
n_liq (dissolved C, mol)  = 0.040437
n_gas (gas-phase C, mol)  = 0.009563
n_liq + n_gas             = 0.050000  (must equal n_total_C = 0.05)


### 5.1  Gotcha: `retain_jacobian=True` refuses a folded gas tableau

The white-box Jacobian machinery (`residual()`, `jacobian_dg_dz()`,
`jacobian_dz_dy()`) keys its internal caches by bare `species_id`, which
collides whenever a gas secondary shares an id with its liquid parent
(the common Henry case — both literally `"CO2"`). Constructing with
`retain_jacobian=True` against a tableau with folded gas secondaries
raises `NotImplementedError` rather than silently producing a wrong
Jacobian; the main `solve()` path above is unaffected. See
`10_engine_protocol_hierarchy.ipynb` for the white-box path against a
non-folded (acid-base-only) tableau.

In [8]:
try:
    NRChemicalEquilibriumEngine.from_reactions(
        [water, co2_first, co2_second, co2_henry], T_K=298.15, retain_jacobian=True,
    )
except NotImplementedError as exc:
    print(f"NotImplementedError: {str(exc)[:120]}...")


NotImplementedError: NRChemicalEquilibriumEngine: retain_jacobian=True is not yet supported for a tableau with folded gas-liquid secondaries ...


## Summary

| Topic | Key takeaway |
|---|---|
| Construction | One flat list; `classify_equilibrium_constraint()` auto-routes acid-base / gas-liquid / solid-liquid |
| `solve()` totals | `totals={master_id: mol/L}` dict + `strong_ions={...}` dict — **not** named kwargs |
| Precipitation | Solid-liquid items auto-detected from the flat list; drives a nested active-set loop; `saturation_indices` / `extra["minerals_xi_mol_L"]` on the result |
| Gas-liquid folding | Fully-parameterized `HenryEquilibrium`/`RaoultEquilibrium` solved *simultaneously*, not diverted; needs `V_liq_L`/`V_gas_L` (or `phases=`); result gains `partial_pressures_atm` |
| `retain_jacobian` | Raises `NotImplementedError` if the tableau has folded gas secondaries (id-collision guard) |
| Result type | `EquilibriumResult`, same as Bisection/PHREEQC; `charge_residual` populated (Bisection always returns `None`) |